# Exercises: Einstein Summation and Index Notation

Practice translating between mathematical notation, explicit loops, and einsum.

**Difficulty levels:** ⭐ Basics | ⭐⭐ Intermediate | ⭐⭐⭐ Advanced

In [ ]:
import numpy as np
np.random.seed(42)

---

## Exercise 1: Basic einsum Operations ⭐

Fill in the einsum subscript strings to perform each operation.

| # | Operation | Expected einsum |
|---|---|---|
| a | Sum all elements of vector | `'???'` |
| b | Dot product of two vectors | `'???'` |
| c | Outer product of two vectors | `'???'` |
| d | Matrix trace | `'???'` |
| e | Matrix transpose | `'???'` |

In [ ]:
# Exercise 1 — Scaffold

a = np.array([1, 2, 3, 4])
b = np.array([5, 6, 7, 8])
M = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])

# (a) Sum all elements: result should be 10
sum_all = np.einsum('???', a)

# (b) Dot product: result should be 70
dot = np.einsum('???', a, b)

# (c) Outer product: result should be (4, 4) matrix
outer = np.einsum('???', a, b)

# (d) Trace of M: result should be 15
trace = np.einsum('???', M)

# (e) Transpose of M
trans = np.einsum('???', M)

print(f'(a) Sum:    {sum_all} (expected 10)')
print(f'(b) Dot:    {dot} (expected 70)')
print(f'(c) Outer shape: {outer.shape} (expected (4, 4))')
print(f'(d) Trace:  {trace} (expected 15)')
print(f'(e) Trans shape: {trans.shape} (expected (3, 3))')

In [ ]:
# Exercise 1 — Solution

a = np.array([1, 2, 3, 4])
b = np.array([5, 6, 7, 8])
M = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])

# (a) 'i->' : i not in output → summed
sum_all = np.einsum('i->', a)

# (b) 'i,i->' : i appears twice → summed (dot product)
dot = np.einsum('i,i->', a, b)

# (c) 'i,j->ij' : no repeated index → outer product
outer = np.einsum('i,j->ij', a, b)

# (d) 'ii->' : diagonal elements, summed
trace = np.einsum('ii->', M)

# (e) 'ij->ji' : swap indices
trans = np.einsum('ij->ji', M)

print(f'(a) Sum:    {sum_all} (expected 10) ✓' if sum_all == 10 else f'(a) Sum: {sum_all} ✗')
print(f'(b) Dot:    {dot} (expected 70) ✓' if dot == 70 else f'(b) Dot: {dot} ✗')
print(f'(c) Outer shape: {outer.shape} ✓')
print(f'(d) Trace:  {trace} (expected 15) ✓' if trace == 15 else f'(d) Trace: {trace} ✗')
print(f'(e) Trans:\n{trans}')
print(f'    Match M.T: {np.array_equal(trans, M.T)} ✓')

---

## Exercise 2: Free vs Dummy Index Identification ⭐

For each einsum expression, identify:
1. Which indices are **free** (appear in output)
2. Which indices are **dummy** (repeated, summed over)
3. What the **output shape** is

Write your answers as comments, then verify with code.

In [ ]:
# Exercise 2 — Scaffold

A = np.random.randn(3, 4)   # shape (3, 4)
B = np.random.randn(4, 5)   # shape (4, 5)
C = np.random.randn(3, 5)   # shape (3, 5)
v = np.random.randn(4)      # shape (4,)

# For each expression, predict BEFORE running:
#   Free indices = ?
#   Dummy indices = ?
#   Output shape = ?

# (a) 'ij,jk->ik'
# Free: ???    Dummy: ???    Shape: ???
r_a = np.einsum('ij,jk->ik', A, B)

# (b) 'ij,jk->ijk'
# Free: ???    Dummy: ???    Shape: ???
r_b = np.einsum('ij,jk->ijk', A, B)

# (c) 'ij,ij->'
# Free: ???    Dummy: ???    Shape: ???
r_c = np.einsum('ij,ij->', A, A)

# (d) 'ij,j->i'
# Free: ???    Dummy: ???    Shape: ???
r_d = np.einsum('ij,j->i', A, v)

# (e) 'ij,ik->jk'
# Free: ???    Dummy: ???    Shape: ???
r_e = np.einsum('ij,ik->jk', A, C)

# Check your predictions:
for label, result in [('a', r_a), ('b', r_b), ('c', r_c), ('d', r_d), ('e', r_e)]:
    print(f'({label}) shape: {result.shape}')

In [ ]:
# Exercise 2 — Solution

print('=== Index Analysis ===')
print()
print('(a) "ij,jk->ik"')
print('    Free: i, k        Dummy: j (summed)')
print('    Shape: (3, 5)  — matrix multiply A(3×4) × B(4×5)')
print(f'    Actual: {r_a.shape}  Match A@B: {np.allclose(r_a, A @ B)}')

print()
print('(b) "ij,jk->ijk"')
print('    Free: i, j, k     Dummy: NONE (j appears in output!)')
print('    Shape: (3, 4, 5)  — no contraction, creates 3D tensor')
print(f'    Actual: {r_b.shape}')

print()
print('(c) "ij,ij->"')
print('    Free: NONE         Dummy: i, j (both summed)')
print('    Shape: ()  — scalar = Frobenius inner product = ‖A‖²_F')
print(f'    Actual: {r_c.shape}  Value: {r_c:.4f}  Match sum(A²): {np.allclose(r_c, np.sum(A**2))}')

print()
print('(d) "ij,j->i"')
print('    Free: i            Dummy: j (summed)')
print('    Shape: (3,)  — matrix-vector multiply A(3×4) × v(4)')
print(f'    Actual: {r_d.shape}  Match A@v: {np.allclose(r_d, A @ v)}')

print()
print('(e) "ij,ik->jk"')
print('    Free: j, k         Dummy: i (summed)')
print('    Shape: (4, 5)  — AᵀC: sum over rows of A and C')
print(f'    Actual: {r_e.shape}  Match A.T@C: {np.allclose(r_e, A.T @ C)}')

---

## Exercise 3: Translate the Math ⭐⭐

Translate each mathematical expression into an `np.einsum` call.

1. **Weighted sum:** $y = \sum_i w_i x_i$
2. **Quadratic form:** $q = \sum_{i,j} x_i A_{ij} x_j = \mathbf{x}^T A \mathbf{x}$
3. **Column-wise sum:** $s_j = \sum_i M_{ij}$ (sum each column)
4. **Hadamard + sum:** $h = \sum_{ij} A_{ij} B_{ij}$ (element-wise multiply then sum all)
5. **Gram matrix:** $G_{ij} = \sum_k X_{ik} X_{jk}$ (i.e., $G = XX^T$)

In [ ]:
# Exercise 3 — Scaffold
np.random.seed(42)

w = np.array([0.1, 0.3, 0.6])
x = np.array([10.0, 20.0, 30.0])
A = np.array([[2, 1], [1, 3]], dtype=float)
x2 = np.array([1.0, 2.0])
M = np.random.randn(3, 4)
B = np.random.randn(3, 4)
X = np.random.randn(5, 3)  # 5 samples, 3 features

# 1. Weighted sum: y = Σᵢ wᵢxᵢ
y = np.einsum('???', w, x)

# 2. Quadratic form: q = xᵢ Aᵢⱼ xⱼ
q = np.einsum('???', x2, A, x2)

# 3. Column-wise sum: sⱼ = Σᵢ Mᵢⱼ
s = np.einsum('???', M)

# 4. Hadamard + sum: h = Aᵢⱼ Bᵢⱼ
h = np.einsum('???', M, B)

# 5. Gram matrix: Gᵢⱼ = Xᵢₖ Xⱼₖ
G = np.einsum('???', X, X)

print(f'1. Weighted sum: {y}')
print(f'2. Quadratic form: {q}')
print(f'3. Column sums: {s}')
print(f'4. Hadamard+sum: {h:.4f}')
print(f'5. Gram matrix shape: {G.shape}')

In [ ]:
# Exercise 3 — Solution
np.random.seed(42)

w = np.array([0.1, 0.3, 0.6])
x = np.array([10.0, 20.0, 30.0])
A = np.array([[2, 1], [1, 3]], dtype=float)
x2 = np.array([1.0, 2.0])
M = np.random.randn(3, 4)
B = np.random.randn(3, 4)
X = np.random.randn(5, 3)

# 1. y = Σᵢ wᵢxᵢ → 'i,i->'  (dot product: i is dummy)
y = np.einsum('i,i->', w, x)
print(f'1. Weighted sum: {y}')
print(f'   Verify: {np.dot(w, x)}  Match: {np.isclose(y, np.dot(w, x))}')

# 2. q = xᵢ Aᵢⱼ xⱼ → 'i,ij,j->'  (both i,j are dummy)
q = np.einsum('i,ij,j->', x2, A, x2)
print(f'\n2. Quadratic form: {q}')
print(f'   Verify: {x2 @ A @ x2}  Match: {np.isclose(q, x2 @ A @ x2)}')

# 3. sⱼ = Σᵢ Mᵢⱼ → 'ij->j'  (i is dummy, j is free)
s = np.einsum('ij->j', M)
print(f'\n3. Column sums: {s.round(3)}')
print(f'   Verify: {M.sum(axis=0).round(3)}  Match: {np.allclose(s, M.sum(axis=0))}')

# 4. h = Aᵢⱼ Bᵢⱼ → 'ij,ij->'  (Frobenius inner product)
h = np.einsum('ij,ij->', M, B)
print(f'\n4. Hadamard+sum: {h:.4f}')
print(f'   Verify: {np.sum(M * B):.4f}  Match: {np.isclose(h, np.sum(M * B))}')

# 5. Gᵢⱼ = Xᵢₖ Xⱼₖ → 'ik,jk->ij'  (k is dummy)
G = np.einsum('ik,jk->ij', X, X)
print(f'\n5. Gram matrix: {G.shape}')
print(f'   Verify XXᵀ match: {np.allclose(G, X @ X.T)}')
print(f'   Symmetric: {np.allclose(G, G.T)}')

---

## Exercise 4: Batch Operations ⭐⭐

In ML, we always have a **batch dimension**. Write einsum expressions for:

1. **Batch dot product:** For each sample in a batch, compute the dot product of two vectors
2. **Batch matrix-vector:** Apply a shared matrix W to each vector in a batch
3. **Batch outer product:** Compute outer product for each pair of vectors in a batch
4. **Batch trace:** Compute the trace of each matrix in a batch

In [ ]:
# Exercise 4 — Scaffold
np.random.seed(42)

batch = 4
a_batch = np.random.randn(batch, 5)    # 4 vectors of dim 5
b_batch = np.random.randn(batch, 5)    # 4 vectors of dim 5
W = np.random.randn(3, 5)             # shared weight matrix (3×5)
x_batch = np.random.randn(batch, 5)    # 4 vectors of dim 5
M_batch = np.random.randn(batch, 3, 3) # 4 square matrices (3×3)

# 1. Batch dot: result shape should be (4,)
dots = np.einsum('???', a_batch, b_batch)

# 2. Batch mat-vec: W × each x. Result shape should be (4, 3)
wx = np.einsum('???', W, x_batch)

# 3. Batch outer: result shape should be (4, 5, 5)
outers = np.einsum('???', a_batch, b_batch)

# 4. Batch trace: result shape should be (4,)
traces = np.einsum('???', M_batch)

print(f'1. Batch dots: {dots.shape}')
print(f'2. Batch mat-vec: {wx.shape}')
print(f'3. Batch outer: {outers.shape}')
print(f'4. Batch trace: {traces.shape}')

In [ ]:
# Exercise 4 — Solution
np.random.seed(42)

batch = 4
a_batch = np.random.randn(batch, 5)
b_batch = np.random.randn(batch, 5)
W = np.random.randn(3, 5)
x_batch = np.random.randn(batch, 5)
M_batch = np.random.randn(batch, 3, 3)

# 1. Batch dot: 'bi,bi->b'  (i is dummy, b is free)
dots = np.einsum('bi,bi->b', a_batch, b_batch)
dots_verify = np.array([np.dot(a_batch[i], b_batch[i]) for i in range(batch)])
print(f'1. Batch dots: {dots.round(3)}')
print(f'   Match: {np.allclose(dots, dots_verify)}')

# 2. Batch mat-vec: 'ij,bj->bi'  (j is dummy, b,i are free)
#    Note: W is NOT batched, but x IS — einsum handles this naturally!
wx = np.einsum('ij,bj->bi', W, x_batch)
wx_verify = np.array([W @ x_batch[i] for i in range(batch)])
print(f'\n2. Batch mat-vec: {wx.shape}')
print(f'   Match: {np.allclose(wx, wx_verify)}')

# 3. Batch outer: 'bi,bj->bij'  (no dummy index → no sum)
outers = np.einsum('bi,bj->bij', a_batch, b_batch)
outers_verify = np.array([np.outer(a_batch[i], b_batch[i]) for i in range(batch)])
print(f'\n3. Batch outer: {outers.shape}')
print(f'   Match: {np.allclose(outers, outers_verify)}')

# 4. Batch trace: 'bii->b'  (i is dummy via diagonal, b is free)
traces = np.einsum('bii->b', M_batch)
traces_verify = np.array([np.trace(M_batch[i]) for i in range(batch)])
print(f'\n4. Batch trace: {traces.round(3)}')
print(f'   Match: {np.allclose(traces, traces_verify)}')

---

## Exercise 5: Attention Score Computation ⭐⭐⭐

Implement multi-head attention scores step-by-step using einsum.

Given:
- Input X of shape (batch, seq_len, d_model)
- Weight matrices W_Q, W_K, W_V of shape (d_model, d_model)

Tasks:
1. Project: Q = X @ W_Q, then reshape to (batch, heads, seq, d_k)
2. Compute attention scores QKᵀ/√d_k using einsum
3. Apply softmax
4. Compute weighted values using einsum

In [ ]:
# Exercise 5 — Scaffold
np.random.seed(42)

batch, seq_len, d_model, n_heads = 2, 8, 16, 4
d_k = d_model // n_heads  # = 4

X = np.random.randn(batch, seq_len, d_model)
W_Q = np.random.randn(d_model, d_model)
W_K = np.random.randn(d_model, d_model)
W_V = np.random.randn(d_model, d_model)

def softmax(x, axis=-1):
    e = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e / np.sum(e, axis=axis, keepdims=True)

# Step 1: Project X to Q, K, V using einsum
# Hint: X(batch, seq, d_model) × W(d_model, d_model) → (batch, seq, d_model)
Q = np.einsum('???', X, W_Q)  # TODO
K = np.einsum('???', X, W_K)  # TODO
V = np.einsum('???', X, W_V)  # TODO

# Step 2: Reshape to multi-head: (batch, seq, d_model) → (batch, heads, seq, d_k)
Q = Q.reshape(batch, seq_len, n_heads, d_k).transpose(0, 2, 1, 3)
K = K.reshape(batch, seq_len, n_heads, d_k).transpose(0, 2, 1, 3)
V = V.reshape(batch, seq_len, n_heads, d_k).transpose(0, 2, 1, 3)

# Step 3: Compute attention scores using einsum
# Q(b,h,i,d) × K(b,h,j,d) → scores(b,h,i,j)
scores = np.einsum('???', Q, K) / np.sqrt(d_k)  # TODO

# Step 4: Softmax
attn_weights = softmax(scores, axis=-1)

# Step 5: Compute output using einsum
# weights(b,h,i,j) × V(b,h,j,d) → output(b,h,i,d)
output = np.einsum('???', attn_weights, V)  # TODO

print(f'Q shape: {Q.shape}')
print(f'Scores shape: {scores.shape}')
print(f'Output shape: {output.shape}')
print(f'Weights sum to 1: {np.allclose(attn_weights.sum(-1), 1.0)}')

In [ ]:
# Exercise 5 — Solution
np.random.seed(42)

batch, seq_len, d_model, n_heads = 2, 8, 16, 4
d_k = d_model // n_heads

X = np.random.randn(batch, seq_len, d_model)
W_Q = np.random.randn(d_model, d_model)
W_K = np.random.randn(d_model, d_model)
W_V = np.random.randn(d_model, d_model)

def softmax(x, axis=-1):
    e = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e / np.sum(e, axis=axis, keepdims=True)

# Step 1: Linear projection
# 'bsd,de->bse' — d is dummy (summed), b,s,e are free
Q = np.einsum('bsd,de->bse', X, W_Q)
K = np.einsum('bsd,de->bse', X, W_K)
V = np.einsum('bsd,de->bse', X, W_V)

print(f'After projection:  Q{Q.shape}')

# Step 2: Reshape to (batch, heads, seq, d_k)
Q = Q.reshape(batch, seq_len, n_heads, d_k).transpose(0, 2, 1, 3)
K = K.reshape(batch, seq_len, n_heads, d_k).transpose(0, 2, 1, 3)
V = V.reshape(batch, seq_len, n_heads, d_k).transpose(0, 2, 1, 3)

print(f'After reshape:     Q{Q.shape}  (batch, heads, seq, d_k)')

# Step 3: Attention scores
# 'bhid,bhjd->bhij' — d is dummy (dot product), b,h,i,j are free
scores = np.einsum('bhid,bhjd->bhij', Q, K) / np.sqrt(d_k)

print(f'Scores:            {scores.shape}  (batch, heads, seq_q, seq_k)')

# Step 4: Softmax (over key dimension j)
attn_weights = softmax(scores, axis=-1)
print(f'Weights sum to 1:  {np.allclose(attn_weights.sum(-1), 1.0)}')

# Step 5: Weighted values
# 'bhij,bhjd->bhid' — j is dummy (weighted sum), b,h,i,d are free
output = np.einsum('bhij,bhjd->bhid', attn_weights, V)

print(f'Output:            {output.shape}  (same layout as Q)')

# Verify against matmul
scores_verify = np.matmul(Q, K.transpose(0, 1, 3, 2)) / np.sqrt(d_k)
output_verify = np.matmul(attn_weights, V)
print(f'\nScores match matmul: {np.allclose(scores, scores_verify)}')
print(f'Output match matmul: {np.allclose(output, output_verify)}')
print('\n→ einsum is more readable: the tensor contraction is EXPLICIT')
print('→ No manual transpose of K needed!')

---

## Exercise 6: Einsum Debugging ⭐⭐

Each of the following einsum calls has a **bug**. Find and fix it.

Common mistakes:
- Wrong number of indices for the tensor rank
- Missing/extra index causing wrong contraction
- Dimension mismatch on shared index

In [ ]:
# Exercise 6 — Scaffold (fix the bugs!)
np.random.seed(42)

A = np.random.randn(3, 4)
B = np.random.randn(4, 5)
v = np.random.randn(4)
M = np.random.randn(3, 3)

# Bug 1: Matrix multiply A(3×4) × B(4×5) — should give (3,5)
# C = np.einsum('ij,jk->jk', A, B)   # ← What's wrong?
# Fix:
# C = np.einsum('???', A, B)

# Bug 2: Matrix-vector: A(3×4) × v(4,) — should give (3,)
# r = np.einsum('ij,i->j', A, v)   # ← What's wrong?
# Fix:
# r = np.einsum('???', A, v)

# Bug 3: Trace of M(3×3) — should give scalar
# t = np.einsum('ij->', M)   # ← What's wrong?
# Fix:
# t = np.einsum('???', M)

# Bug 4: Dot product of v with itself — should give scalar
# d = np.einsum('i,j->', v, v)   # ← What's wrong?
# Fix:
# d = np.einsum('???', v, v)

print('Uncomment the Fix lines above and run!')

In [ ]:
# Exercise 6 — Solution
np.random.seed(42)

A = np.random.randn(3, 4)
B = np.random.randn(4, 5)
v = np.random.randn(4)
M = np.random.randn(3, 3)

# Bug 1: 'ij,jk->jk' — output has j, but j should be contracted!
#   This keeps j (size 4) and k (size 5), giving (4,5) not (3,5)
# Fix: j must NOT be in output
C = np.einsum('ij,jk->ik', A, B)
print(f'Bug 1 fixed: shape {C.shape}, match A@B: {np.allclose(C, A @ B)}')

# Bug 2: 'ij,i->j' — v is 1D (rank 1) but gets index i (size 3)
#   v has 4 elements, A's first dim is 3 → dimension mismatch on i
# Fix: v should align with A's SECOND dim (j)
r = np.einsum('ij,j->i', A, v)
print(f'Bug 2 fixed: shape {r.shape}, match A@v: {np.allclose(r, A @ v)}')

# Bug 3: 'ij->' — this sums ALL elements, not just the diagonal
#   Trace = sum of diagonal = Σᵢ Mᵢᵢ, needs repeated index
# Fix: use 'ii->' to select diagonal then sum
t = np.einsum('ii->', M)
print(f'Bug 3 fixed: trace = {t:.4f}, match np.trace: {np.isclose(t, np.trace(M))}')

# Bug 4: 'i,j->' — uses DIFFERENT indices for same vector
#   This computes Σᵢ Σⱼ vᵢwⱼ = (Σᵢ vᵢ)(Σⱼ wⱼ) = sum(v) × sum(v)
#   That's the product of sums, NOT the sum of products (dot product)
# Fix: use SAME index → repeated index triggers dot product
d = np.einsum('i,i->', v, v)
print(f'Bug 4 fixed: dot = {d:.4f}, match np.dot: {np.isclose(d, np.dot(v, v))}')

# Demonstrate Bug 4 misunderstanding
wrong = np.einsum('i,j->', v, v)  # = sum(v)²
right = np.einsum('i,i->', v, v)  # = v·v
print(f'\nBug 4 deep dive:')
print(f'  "i,j->" gives {wrong:.4f} = sum(v)² = ({v.sum():.4f})² = {v.sum()**2:.4f}')
print(f'  "i,i->" gives {right:.4f} = v·v = ‖v‖² = {np.sum(v**2):.4f}')

---

## Exercise 7: Write It Three Ways ⭐⭐⭐

For each operation, implement it three ways and verify they match:
1. **Explicit loops** (for understanding)
2. **NumPy functions** (dot, @, sum, etc.)
3. **einsum**

Operations:
- (a) Batch matrix multiply: A(b,m,n) × B(b,n,p) → C(b,m,p)
- (b) Bilinear form: x(m,) W(m,n) y(n,) → scalar

In [ ]:
# Exercise 7 — Scaffold
np.random.seed(42)

# (a) Batch matmul
batch, m, n, p = 3, 4, 5, 6
A = np.random.randn(batch, m, n)
B = np.random.randn(batch, n, p)

# Loop version:
C_loop = np.zeros((batch, m, p))
# TODO: fill with loops

# NumPy version:
# C_numpy = ???

# einsum version:
# C_einsum = np.einsum('???', A, B)

# (b) Bilinear form
x = np.random.randn(4)
W = np.random.randn(4, 3)
y = np.random.randn(3)

# Loop version:
s_loop = 0.0
# TODO: fill with loops

# NumPy version:
# s_numpy = ???

# einsum version:
# s_einsum = np.einsum('???', x, W, y)

In [ ]:
# Exercise 7 — Solution
np.random.seed(42)

# ═══ (a) Batch matmul ═══
batch, m, n, p = 3, 4, 5, 6
A = np.random.randn(batch, m, n)
B = np.random.randn(batch, n, p)

# Loop version
C_loop = np.zeros((batch, m, p))
for b in range(batch):
    for i in range(m):
        for j in range(p):
            for k in range(n):  # k is the contracted index
                C_loop[b, i, j] += A[b, i, k] * B[b, k, j]

# NumPy version
C_numpy = np.matmul(A, B)  # or A @ B

# einsum version
C_einsum = np.einsum('bik,bkj->bij', A, B)

print('(a) Batch matmul:')
print(f'  Loop == NumPy:  {np.allclose(C_loop, C_numpy)}')
print(f'  Loop == einsum: {np.allclose(C_loop, C_einsum)}')

# ═══ (b) Bilinear form ═══
x = np.random.randn(4)
W = np.random.randn(4, 3)
y = np.random.randn(3)

# Loop version
s_loop = 0.0
for i in range(len(x)):
    for j in range(len(y)):
        s_loop += x[i] * W[i, j] * y[j]

# NumPy version
s_numpy = x @ W @ y

# einsum version
s_einsum = np.einsum('i,ij,j->', x, W, y)

print(f'\n(b) Bilinear form xᵀWy:')
print(f'  Loop:   {s_loop:.6f}')
print(f'  NumPy:  {s_numpy:.6f}')
print(f'  einsum: {s_einsum:.6f}')
print(f'  All match: {np.allclose(s_loop, s_numpy) and np.allclose(s_numpy, s_einsum)}')

---

## Exercise 8: PyTorch einsum ⭐⭐⭐

Rewrite the NumPy attention implementation from Exercise 5 using `torch.einsum` and verify it matches `torch.nn.functional.scaled_dot_product_attention` (or manual matmul).

In [ ]:
# Exercise 8 — Scaffold
import torch
import torch.nn.functional as F
torch.manual_seed(42)

batch, heads, seq_len, d_k = 2, 4, 8, 16

Q = torch.randn(batch, heads, seq_len, d_k)
K = torch.randn(batch, heads, seq_len, d_k)
V = torch.randn(batch, heads, seq_len, d_k)

# TODO: Compute attention scores with torch.einsum
# scores = torch.einsum('???', Q, K) / (d_k ** 0.5)

# TODO: Softmax
# attn_weights = F.softmax(scores, dim=-1)

# TODO: Compute output with torch.einsum
# output = torch.einsum('???', attn_weights, V)

# Verify:
# output_matmul = torch.matmul(attn_weights, V)
# print(f'Match: {torch.allclose(output, output_matmul)}')

In [ ]:
# Exercise 8 — Solution
import torch
import torch.nn.functional as F
import math
torch.manual_seed(42)

batch, heads, seq_len, d_k = 2, 4, 8, 16

Q = torch.randn(batch, heads, seq_len, d_k)
K = torch.randn(batch, heads, seq_len, d_k)
V = torch.randn(batch, heads, seq_len, d_k)

# Attention scores via einsum
# 'bhid,bhjd->bhij' — d is contracted (dot product per query-key pair)
scores = torch.einsum('bhid,bhjd->bhij', Q, K) / math.sqrt(d_k)
print(f'Scores shape: {scores.shape}')

# Softmax over key dimension (last dim = j)
attn_weights = F.softmax(scores, dim=-1)
print(f'Weights sum to 1: {torch.allclose(attn_weights.sum(-1), torch.ones(batch, heads, seq_len))}')

# Weighted values via einsum
# 'bhij,bhjd->bhid' — j is contracted (weighted sum over values)
output = torch.einsum('bhij,bhjd->bhid', attn_weights, V)
print(f'Output shape: {output.shape}')

# Verify against matmul
scores_matmul = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
output_matmul = torch.matmul(F.softmax(scores_matmul, dim=-1), V)

print(f'\nScores match matmul: {torch.allclose(scores, scores_matmul)}')
print(f'Output match matmul: {torch.allclose(output, output_matmul, atol=1e-6)}')

# Compare readability
print('\n--- Readability comparison ---')
print('einsum:  scores = torch.einsum("bhid,bhjd->bhij", Q, K)')
print('matmul:  scores = torch.matmul(Q, K.transpose(-2, -1))')
print('\n→ einsum:  you can READ the contraction (d summed, i×j grid)')
print('→ matmul:  requires knowing transpose convention')

---

## Exercise 9: Contraction Order Challenge ⭐⭐⭐

Given three tensors A(100×10), B(10×100), C(100×5):

1. Compute the FLOP count for (AB)C vs A(BC)
2. Time both orderings
3. Use `np.einsum` with `optimize=True` and compare

In [ ]:
# Exercise 9 — Scaffold
np.random.seed(42)
import time

m, n, p, q = 100, 10, 100, 5
A = np.random.randn(m, n)   # 100×10
B = np.random.randn(n, p)   # 10×100
C = np.random.randn(p, q)   # 100×5

# 1. FLOP counts:
# (AB)C: AB is (m×n)×(n×p) = ??? FLOPs, then ×C is (m×p)×(p×q) = ??? FLOPs
flops_abc = 0  # TODO: m*n*p + ???

# A(BC): BC is (n×p)×(p×q) = ??? FLOPs, then A× is (m×n)×(n×q) = ??? FLOPs
flops_a_bc = 0  # TODO: n*p*q + ???

print(f'(AB)C FLOPs: {flops_abc:,}')
print(f'A(BC) FLOPs: {flops_a_bc:,}')
print(f'Ratio: {flops_abc/flops_a_bc:.1f}x')

# 2. Time both
# TODO

# 3. einsum with optimize=True
# TODO

In [ ]:
# Exercise 9 — Solution
np.random.seed(42)
import time

m, n, p, q = 100, 10, 100, 5
A = np.random.randn(m, n)   # 100×10
B = np.random.randn(n, p)   # 10×100
C = np.random.randn(p, q)   # 100×5

# 1. FLOP analysis
# (AB)C: AB is m×n times n×p = m·n·p mults, then (m×p)×(p×q) = m·p·q
flops_abc = m*n*p + m*p*q  # 100·10·100 + 100·100·5 = 100,000 + 50,000
# A(BC): BC is n×p times p×q = n·p·q, then (m×n)×(n×q) = m·n·q
flops_a_bc = n*p*q + m*n*q  # 10·100·5 + 100·10·5 = 5,000 + 5,000

print(f'=== FLOP Analysis ===')
print(f'(AB)C: {m}·{n}·{p} + {m}·{p}·{q} = {flops_abc:,}')
print(f'A(BC): {n}·{p}·{q} + {m}·{n}·{q} = {flops_a_bc:,}')
print(f'Ratio: {flops_abc/flops_a_bc:.1f}×  — A(BC) is cheaper!')

# 2. Timing
trials = 5000

t0 = time.perf_counter()
for _ in range(trials):
    r1 = (A @ B) @ C
t_abc = (time.perf_counter() - t0) / trials

t0 = time.perf_counter()
for _ in range(trials):
    r2 = A @ (B @ C)
t_a_bc = (time.perf_counter() - t0) / trials

print(f'\n=== Timing ===')
print(f'(AB)C:  {t_abc*1e6:.1f} μs')
print(f'A(BC):  {t_a_bc*1e6:.1f} μs')
print(f'Speedup: {t_abc/t_a_bc:.1f}×')
print(f'Results match: {np.allclose(r1, r2)}')

# 3. einsum with optimize=True
t0 = time.perf_counter()
for _ in range(trials):
    r3 = np.einsum('ij,jk,kl->il', A, B, C, optimize=True)
t_opt = (time.perf_counter() - t0) / trials

print(f'\n=== einsum(optimize=True) ===')
print(f'Time:  {t_opt*1e6:.1f} μs')
print(f'Match: {np.allclose(r1, r3)}')

# Show the contraction path that optimize finds
path, info = np.einsum_path('ij,jk,kl->il', A, B, C, optimize='optimal')
print(f'\nOptimal contraction order: {path}')
print(f'→ optimize=True automatically finds the cheapest order')

---

## Summary

| Exercise | Skill Practiced |
|---|---|
| 1. Basic einsum | Writing simple subscript strings |
| 2. Index identification | Distinguishing free vs dummy indices |
| 3. Translate math | Converting Σ notation to einsum |
| 4. Batch operations | Adding batch dimensions |
| 5. Attention scores | Full multi-head attention pipeline |
| 6. Debugging | Finding common einsum bugs |
| 7. Three ways | Loop ↔ NumPy ↔ einsum equivalence |
| 8. PyTorch einsum | torch.einsum for real models |
| 9. Contraction order | FLOP analysis and optimization |